# Pipeline Data Explorer

Reads data from all three zones in the `data-pipeline` MinIO bucket using PySpark (local mode).

| Zone | MinIO Path | Written by |
|------|-----------|------------|
| Raw | `s3a://data-pipeline/raw/` | DAG 01 — 01_generate_ingest.py |
| Processed | `s3a://data-pipeline/processed/` | DAG 02 — 02_process_data.py |
| Aggregated | `s3a://data-pipeline/aggregated/` | DAG 03 — 03_aggregate.py |

**Run cells in order.** The first cell initialises MinIO zone structure and reports what data is available.

## 0 — Initialize & Status

Creates zone prefixes in MinIO (if they don't exist) and reports which zones have data.

In [ ]:
import os
import boto3
from botocore.exceptions import ClientError

MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT',   'http://minio:9000')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY', 'minioadmin')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY', 'minioadmin123')
MINIO_BUCKET     = os.getenv('MINIO_BUCKET',     'data-pipeline')

s3 = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    region_name='us-east-1',
)

# Ensure the bucket exists
try:
    s3.head_bucket(Bucket=MINIO_BUCKET)
except ClientError:
    s3.create_bucket(Bucket=MINIO_BUCKET)
    print(f'Created bucket: {MINIO_BUCKET}')

# Zone definitions: name -> expected prefix
ZONES = {
    'raw':              'raw/',
    'staging/clean':    'staging/clean/',
    'staging/enriched': 'staging/enriched/',
    'processed':        'processed/',
    'aggregated':       'aggregated/',
    'pipeline-logs':    None,  # separate bucket check below
}

# Create a zero-byte zone marker so the prefix always exists in MinIO
MARKER = '_zone_initialized'
for zone, prefix in ZONES.items():
    if prefix is None:
        continue
    key = f'{prefix}{MARKER}'
    try:
        s3.head_object(Bucket=MINIO_BUCKET, Key=key)
    except ClientError:
        s3.put_object(Bucket=MINIO_BUCKET, Key=key, Body=b'')
        print(f'  Initialized zone: s3://{MINIO_BUCKET}/{prefix}')

# Ensure pipeline-logs bucket exists too
for bucket in ['pipeline-logs']:
    try:
        s3.head_bucket(Bucket=bucket)
    except ClientError:
        s3.create_bucket(Bucket=bucket)
        print(f'Created bucket: {bucket}')

print()
print('=== Zone status ===')

def zone_has_parquet(prefix):
    """Return (file_count, total_mb) for .parquet files under prefix."""
    resp = s3.list_objects_v2(Bucket=MINIO_BUCKET, Prefix=prefix)
    objects = [o for o in resp.get('Contents', []) if o['Key'].endswith('.parquet')]
    return len(objects), sum(o['Size'] for o in objects) / 1024 / 1024

ZONE_PATHS = {
    'raw':              'raw/',
    'staging/clean':    'staging/clean/',
    'staging/enriched': 'staging/enriched/',
    'processed':        'processed/',
    'agg/daily_summary':    'aggregated/daily_summary/',
    'agg/category_ranking': 'aggregated/category_ranking/',
    'agg/hourly_trends':    'aggregated/hourly_trends/',
}

ZONE_STATUS = {}
for name, prefix in ZONE_PATHS.items():
    count, mb = zone_has_parquet(prefix)
    has_data = count > 0
    ZONE_STATUS[name] = has_data
    status = f'{count:>3} parquet files  {mb:>7.2f} MB' if has_data else 'no data yet'
    icon   = '✓' if has_data else '○'
    print(f'  {icon}  {name:<28}  {status}')

print()
print('Run DAG 01 to populate raw, DAG 02 for processed, DAG 03 for aggregated.')

## 1 — SparkSession (local mode → reads directly from MinIO via S3A)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Use pre-baked JARs from image build (no Ivy download at runtime)
import pathlib
jars_txt = pathlib.Path('/opt/spark-jars/jars.txt')
SPARK_JARS = jars_txt.read_text().strip() if jars_txt.exists() else None

builder = (
    SparkSession.builder
    .master('local[*]')
    .appName('JupyterExplorer')
    .config('spark.jars.ivy',                             '/tmp/.ivy')
    .config('spark.hadoop.fs.s3a.endpoint',               MINIO_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key',             MINIO_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key',             MINIO_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access',      'true')
    .config('spark.hadoop.fs.s3a.impl',
            'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .config('spark.sql.adaptive.enabled',                 'true')
    .config('spark.ui.showConsoleProgress',               'false')
)

if SPARK_JARS:
    builder = builder.config('spark.jars', SPARK_JARS)
else:
    builder = builder.config(
        'spark.jars.packages',
        'org.apache.hadoop:hadoop-aws:3.4.0,'
        'com.amazonaws:aws-java-sdk-bundle:1.12.262'
    )

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel('WARN')
print(f'Spark {spark.version}  master={spark.sparkContext.master}')
print(f'MinIO endpoint: {MINIO_ENDPOINT}')

## Helper — safe zone reader

Returns `None` (with a message) when a zone has no data yet, so downstream cells don't crash.

In [ ]:
from py4j.protocol import Py4JJavaError

def safe_read_parquet(path, label=None):
    """Read a Parquet path from MinIO; return None if the path has no data yet."""
    label = label or path
    try:
        df = spark.read.parquet(path)
        # Force schema resolution — triggers the FileNotFoundException early
        _ = df.schema
        return df
    except (Py4JJavaError, Exception) as e:
        msg = str(e)
        if 'FileNotFoundException' in msg or 'NoSuchKey' in msg or 'not exist' in msg.lower():
            print(f'  [skip] {label} — no data yet. Run the corresponding DAG first.')
            return None
        raise

## 2 — Browse MinIO buckets (boto3)

In [ ]:
print('=== MinIO buckets ===')
for b in s3.list_buckets()['Buckets']:
    resp  = s3.list_objects_v2(Bucket=b['Name'])
    count = resp.get('KeyCount', 0)
    size  = sum(o['Size'] for o in resp.get('Contents', []))
    print(f"  {b['Name']:<20}  {count:>4} objects  {size/1024/1024:>8.1f} MB")

print()
print(f'=== {MINIO_BUCKET}: top-level zones ===')
resp = s3.list_objects_v2(Bucket=MINIO_BUCKET, Delimiter='/')
for p in resp.get('CommonPrefixes', []):
    print(f"  {p['Prefix']}")

## 3 — Raw zone

Synthetic orders generated by DAG 01. Partitioned by `year / month / day`.

In [ ]:
raw_df = safe_read_parquet(f's3a://{MINIO_BUCKET}/raw/', 'raw zone')
if raw_df is not None:
    # Drop duplicate partition cols Spark infers from both the path and the data
    for col in ['year', 'month', 'day']:
        if raw_df.columns.count(col) > 1:
            raw_df = raw_df.drop(col).withColumnRenamed(f'{col}#1', col)
    print(f'Rows      : {raw_df.count():,}')
    print(f'Partitions: {raw_df.rdd.getNumPartitions()}')
    raw_df.printSchema()

In [ ]:
if raw_df is not None:
    raw_df.show(5, truncate=False)

In [ ]:
if raw_df is not None:
    # Revenue by category
    raw_df.groupBy('category').agg(
        F.count('order_id').alias('orders'),
        F.round(F.sum('revenue'), 2).alias('total_revenue'),
        F.round(F.avg('revenue'), 2).alias('avg_order_value'),
        F.round(F.avg('discount_pct') * 100, 1).alias('avg_discount_%'),
    ).orderBy(F.col('total_revenue').desc()).show()

## 4 — Processed zone

Cleaned and enriched by DAG 02 (clean → enrich → write). Partitioned by `category / year / month / day`.

In [ ]:
proc_df = safe_read_parquet(f's3a://{MINIO_BUCKET}/processed/', 'processed zone')
if proc_df is not None:
    print(f'Rows      : {proc_df.count():,}')
    proc_df.printSchema()

In [ ]:
if proc_df is not None:
    # Enriched columns added by DAG 02
    proc_df.select(
        'order_id', 'category', 'price_tier', 'revenue_band',
        'is_discounted', 'gross_margin_est', 'event_hour',
        'is_weekend', 'country_name',
    ).show(10)

In [ ]:
if proc_df is not None:
    # Revenue by price tier
    proc_df.groupBy('price_tier').agg(
        F.count('*').alias('orders'),
        F.round(F.sum('revenue'), 2).alias('total_revenue'),
        F.round(F.avg('gross_margin_est'), 2).alias('avg_margin'),
    ).orderBy('total_revenue', ascending=False).show()

In [ ]:
if proc_df is not None:
    # Orders by hour of day
    proc_df.groupBy('event_hour').agg(
        F.count('*').alias('orders'),
        F.round(F.avg('revenue'), 2).alias('avg_revenue'),
    ).orderBy('event_hour').show(24)

## 5 — Aggregated zone

Three parallel aggregations written by DAG 03.

In [ ]:
# Daily summary — revenue + orders grouped by category × region × channel × price_tier
daily_df = safe_read_parquet(
    f's3a://{MINIO_BUCKET}/aggregated/daily_summary/', 'daily_summary'
)
if daily_df is not None:
    print(f'daily_summary rows: {daily_df.count():,}')
    daily_df.orderBy(F.col('total_revenue').desc()).show(10)

In [ ]:
# Category ranking — month-to-date revenue rank + % share
rank_df = safe_read_parquet(
    f's3a://{MINIO_BUCKET}/aggregated/category_ranking/', 'category_ranking'
)
if rank_df is not None:
    print(f'category_ranking rows: {rank_df.count():,}')
    rank_df.orderBy('revenue_rank').show()

In [ ]:
# Hourly trends — orders + avg revenue by hour with cumulative revenue by channel
hourly_df = safe_read_parquet(
    f's3a://{MINIO_BUCKET}/aggregated/hourly_trends/', 'hourly_trends'
)
if hourly_df is not None:
    print(f'hourly_trends rows: {hourly_df.count():,}')
    hourly_df.orderBy('event_hour').show(24)

## 6 — Visualisations

Cells below skip automatically when the required zone has no data yet.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('dark_background')
sns.set_palette('husl')

In [ ]:
if proc_df is not None:
    # Revenue by category
    cat_pd = (
        proc_df.groupBy('category')
        .agg(F.round(F.sum('revenue') / 1e6, 2).alias('revenue_M'))
        .orderBy(F.col('revenue_M').desc())
        .toPandas()
    )

    fig, ax = plt.subplots(figsize=(9, 4))
    colors = sns.color_palette('husl', len(cat_pd))
    bars = ax.barh(cat_pd['category'], cat_pd['revenue_M'], color=colors)
    ax.bar_label(bars, fmt='$%.2fM', padding=4, color='white', fontsize=9)
    ax.set_xlabel('Revenue (millions $)')
    ax.set_title('Total Revenue by Category', fontsize=13, fontweight='bold')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print('[skip] Processed zone not ready — run DAG 02 first.')

In [ ]:
if hourly_df is not None:
    # Orders + avg order value by hour of day
    hourly_pd = (
        hourly_df.groupBy('event_hour')
        .agg(F.sum('order_count').alias('orders'),
             F.round(F.avg('avg_order_value'), 2).alias('avg_value'))
        .orderBy('event_hour')
        .toPandas()
    )

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

    ax1.plot(hourly_pd['event_hour'], hourly_pd['orders'], marker='o', linewidth=2, color='#7c91f5')
    ax1.fill_between(hourly_pd['event_hour'], hourly_pd['orders'], alpha=0.2, color='#7c91f5')
    ax1.set_ylabel('Order Count')
    ax1.set_title('Orders & Avg Order Value by Hour of Day', fontsize=13, fontweight='bold')
    ax1.grid(alpha=0.2)

    ax2.plot(hourly_pd['event_hour'], hourly_pd['avg_value'], marker='s', linewidth=2, color='#4ade80')
    ax2.fill_between(hourly_pd['event_hour'], hourly_pd['avg_value'], alpha=0.2, color='#4ade80')
    ax2.set_xlabel('Hour of Day')
    ax2.set_ylabel('Avg Order Value ($)')
    ax2.set_xticks(range(0, 24))
    ax2.grid(alpha=0.2)

    plt.tight_layout()
    plt.show()
else:
    print('[skip] hourly_trends not ready — run DAG 03 first.')

In [ ]:
if proc_df is not None:
    # Revenue share by price tier + revenue by country
    tier_pd = (
        proc_df.groupBy('price_tier')
        .agg(F.sum('revenue').alias('revenue'))
        .orderBy(F.col('revenue').desc())
        .toPandas()
    )
    country_pd = (
        proc_df.groupBy('country_name')
        .agg(F.round(F.sum('revenue') / 1e6, 2).alias('revenue_M'))
        .orderBy(F.col('revenue_M').desc())
        .toPandas()
    )

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    colors = sns.color_palette('husl', len(tier_pd))
    ax1.pie(tier_pd['revenue'], labels=tier_pd['price_tier'], autopct='%1.1f%%',
            colors=colors, startangle=140, textprops={'fontsize': 11})
    ax1.set_title('Revenue Share by Price Tier', fontweight='bold')

    bars = ax2.bar(country_pd['country_name'], country_pd['revenue_M'],
                   color=sns.color_palette('husl', len(country_pd)))
    ax2.bar_label(bars, fmt='$%.1fM', padding=2, fontsize=9)
    ax2.set_title('Revenue by Country', fontweight='bold')
    ax2.set_ylabel('Revenue (millions $)')
    ax2.tick_params(axis='x', rotation=30)

    plt.tight_layout()
    plt.show()
else:
    print('[skip] Processed zone not ready — run DAG 02 first.')

In [ ]:
if proc_df is not None:
    # Order count heatmap: category × revenue_band
    heat_pd = (
        proc_df.groupBy('category', 'revenue_band')
        .agg(F.count('*').alias('orders'))
        .toPandas()
        .pivot(index='category', columns='revenue_band', values='orders')
        .fillna(0)
    )
    col_order = [c for c in ['Low', 'Medium', 'High', 'Very High'] if c in heat_pd.columns]
    heat_pd = heat_pd[col_order]

    fig, ax = plt.subplots(figsize=(9, 4))
    sns.heatmap(heat_pd, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax,
                linewidths=0.5, cbar_kws={'label': 'Order Count'})
    ax.set_title('Order Count — Category × Revenue Band', fontsize=13, fontweight='bold')
    ax.set_xlabel('Revenue Band')
    ax.set_ylabel('')
    plt.tight_layout()
    plt.show()
else:
    print('[skip] Processed zone not ready — run DAG 02 first.')

## 7 — Ad-hoc Spark SQL

In [ ]:
views = []
if raw_df   is not None: raw_df.createOrReplaceTempView('raw_orders');      views.append('raw_orders')
if proc_df  is not None: proc_df.createOrReplaceTempView('processed_orders'); views.append('processed_orders')
if daily_df is not None: daily_df.createOrReplaceTempView('daily_summary');  views.append('daily_summary')
print('Temp views ready:', ', '.join(views) if views else 'none — run DAGs first')

In [ ]:
if proc_df is not None:
    spark.sql("""
        SELECT
            category,
            channel,
            COUNT(*)                            AS orders,
            ROUND(SUM(revenue), 2)              AS total_revenue,
            ROUND(AVG(discount_pct) * 100, 1)  AS avg_discount_pct
        FROM processed_orders
        GROUP BY category, channel
        ORDER BY total_revenue DESC
        LIMIT 15
    """).show()

In [ ]:
if proc_df is not None:
    # Write your own query here
    spark.sql("""
        SELECT order_id, category, sub_category, country_name,
               quantity, unit_price, revenue, price_tier
        FROM processed_orders
        ORDER BY revenue DESC
        LIMIT 10
    """).show(truncate=False)

In [ ]:
spark.stop()
print('Session stopped.')